# 🌱 Plant Disease Detection — Training Notebook

**Before running anything:**
1. Go to **Runtime → Change runtime type → T4 GPU → Save**
2. Then run all cells top to bottom with **Runtime → Run all**

---
| Step | Cell | Time estimate |
|------|------|---------------|
| Install packages | Cell 1 | ~1 min |
| Upload dataset zip | Cell 2 | depends on internet |
| Setup project | Cell 3 | ~30 sec |
| Train MobileNetV2 | Cell 4 | ~30–45 min on T4 |
| Evaluate & plots | Cell 5 | ~5 min |
| Download model | Cell 6 | instant |

## Cell 1 — Install Dependencies

In [ ]:
# All packages needed — torch is already on Colab, just need extras
!pip install -q opencv-python-headless scikit-learn tqdm

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'GPU available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name        : {torch.cuda.get_device_name(0)}')
    print(f'GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

## Cell 2 — Upload Dataset

**Option A (recommended): Upload from your computer**  
Run the cell below. A file picker will appear. Upload the zip file you downloaded from Kaggle.

**Option B: Download directly from Kaggle API**  
Skip to Cell 2b if you have a kaggle.json API key.

In [ ]:
# ── OPTION A: Upload zip from your computer ──────────────
from google.colab import files
import os, zipfile, shutil
from pathlib import Path

DATA_DIR = Path('/content/data/PlantVillage')

if DATA_DIR.exists() and any(DATA_DIR.iterdir()):
    count = sum(1 for _ in DATA_DIR.rglob('*.jpg')) + sum(1 for _ in DATA_DIR.rglob('*.JPG'))
    print(f'✅ Dataset already uploaded! Found {count:,} images in {DATA_DIR}')
else:
    print('📁 Select your Kaggle zip file to upload...')
    print('   (The file is usually named: archive.zip or plantvillage-dataset.zip)')
    uploaded = files.upload()

    zip_name = list(uploaded.keys())[0]
    print(f'\n📦 Extracting {zip_name}...')

    os.makedirs('/content/data/raw', exist_ok=True)
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('/content/data/raw')

    # Find the 'color' subfolder (contains the RGB images we want)
    color_dir = None
    for root, dirs, files_list in os.walk('/content/data/raw'):
        for d in dirs:
            if d.lower() == 'color':
                color_dir = Path(root) / d
                break
        if color_dir:
            break

    if color_dir is None:
        # Maybe the classes are already at the top level
        print('  color/ folder not found — scanning for class folders directly...')
        for root, dirs, _ in os.walk('/content/data/raw'):
            if any('__' in d for d in dirs):
                color_dir = Path(root)
                break

    if color_dir:
        DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(str(color_dir), str(DATA_DIR))
        count = sum(1 for _ in DATA_DIR.rglob('*.jpg')) + sum(1 for _ in DATA_DIR.rglob('*.JPG'))
        classes = [d for d in DATA_DIR.iterdir() if d.is_dir()]
        print(f'\n✅ Done! {count:,} images across {len(classes)} classes')
        print(f'   Saved to: {DATA_DIR}')
    else:
        print('⚠️  Could not auto-detect class folders.')
        print('   Contents of /content/data/raw:')
        for p in sorted(Path('/content/data/raw').rglob('*'))[:30]:
            print(f'   {p}')

In [ ]:
# ── OPTION B: Download directly from Kaggle API ──────────
# Only run this cell if you skipped Option A above.
# You need a kaggle.json API key from https://www.kaggle.com/settings

RUN_OPTION_B = False   # ← Set to True to use Kaggle API

if RUN_OPTION_B:
    from google.colab import files
    print('Upload your kaggle.json file...')
    files.upload()   # upload kaggle.json

    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    !pip install -q kaggle
    !kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/data --unzip

    import shutil
    from pathlib import Path
    color_dir = Path('/content/data/plantvillage dataset/color')
    DATA_DIR = Path('/content/data/PlantVillage')
    if color_dir.exists():
        shutil.copytree(str(color_dir), str(DATA_DIR))
        print(f'✅ Dataset ready at {DATA_DIR}')
    else:
        print('Adjust the color_dir path above to match what was downloaded.')

## Cell 3 — Project Setup (Config, Model, Data Pipeline)

In [ ]:
# ── Config ───────────────────────────────────────────────
from pathlib import Path

DATA_DIR    = Path('/content/data/PlantVillage')
MODEL_DIR   = Path('/content/models')
OUTPUT_DIR  = Path('/content/outputs')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE        = 224
BATCH_SIZE      = 64      # T4 has 16GB — 64 is safe and fast
NUM_WORKERS     = 2
LEARNING_RATE   = 1e-3
FINETUNE_LR     = 1e-5
NUM_EPOCHS      = 20
PATIENCE        = 5
RANDOM_SEED     = 42
TRAIN_RATIO     = 0.70
VAL_RATIO       = 0.15
IMAGENET_MEAN   = [0.485, 0.456, 0.406]
IMAGENET_STD    = [0.229, 0.224, 0.225]
MODEL_NAME      = 'mobilenetv2'   # ← change to 'resnet50' if you want

print(f'Config loaded. Model: {MODEL_NAME}, Batch size: {BATCH_SIZE}')

In [ ]:
# ── Imports ──────────────────────────────────────────────
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.notebook import tqdm

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ── Helper: plant/condition name splitter ─────────────────
def get_plant_and_condition(class_name):
    if '___' in class_name:
        parts = class_name.split('___', 1)
    elif '__' in class_name:
        parts = class_name.split('__', 1)
    else:
        return class_name.replace('_', ' '), 'Unknown'
    plant     = parts[0].replace('_', ' ').strip()
    condition = parts[1].replace('_', ' ').strip() if len(parts) > 1 else 'Unknown'
    return plant, condition

print('Helper functions ready.')

In [ ]:
# ── Data Pipeline ────────────────────────────────────────
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Load full dataset to get targets for stratified splitting
full_ds       = datasets.ImageFolder(DATA_DIR, transform=val_transforms)
class_to_idx  = full_ds.class_to_idx
idx_to_class  = {v: k for k, v in class_to_idx.items()}
NUM_CLASSES   = len(class_to_idx)
targets       = full_ds.targets
indices       = list(range(len(full_ds)))

print(f'Total images : {len(full_ds):,}')
print(f'Num classes  : {NUM_CLASSES}')

# Stratified split
train_idx, valtest_idx = train_test_split(
    indices, test_size=1-TRAIN_RATIO,
    stratify=targets, random_state=RANDOM_SEED
)
valtest_targets = [targets[i] for i in valtest_idx]
relative_test   = VAL_RATIO / (1 - TRAIN_RATIO)
val_idx, test_idx = train_test_split(
    valtest_idx, test_size=relative_test,
    stratify=valtest_targets, random_state=RANDOM_SEED
)
print(f'Train: {len(train_idx):,}  |  Val: {len(val_idx):,}  |  Test: {len(test_idx):,}')

# Weighted sampler to fix class imbalance
train_targets   = [targets[i] for i in train_idx]
class_counts    = Counter(train_targets)
sample_weights  = torch.tensor(
    [1.0 / class_counts[targets[i]] for i in train_idx], dtype=torch.float
)
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_idx), replacement=True)
print(f'WeightedRandomSampler active — imbalance handled')

# Build datasets with correct transforms
train_ds = datasets.ImageFolder(DATA_DIR, transform=train_transforms)
val_ds   = datasets.ImageFolder(DATA_DIR, transform=val_transforms)

loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=True)
train_loader  = DataLoader(Subset(train_ds, train_idx), batch_size=BATCH_SIZE, sampler=sampler, **loader_kwargs)
val_loader    = DataLoader(Subset(val_ds,   val_idx),   batch_size=BATCH_SIZE, shuffle=False,   **loader_kwargs)
test_loader   = DataLoader(Subset(val_ds,   test_idx),  batch_size=BATCH_SIZE, shuffle=False,   **loader_kwargs)

# Save class mapping
with open(MODEL_DIR / 'class_mapping.json', 'w') as f:
    json.dump(idx_to_class, f, indent=2)
print(f'Class mapping saved → models/class_mapping.json')

In [ ]:
# ── Model Definition ─────────────────────────────────────
def build_model(model_name, num_classes, pretrained=True, freeze_backbone=True):
    if model_name == 'mobilenetv2':
        weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
        model   = models.mobilenet_v2(weights=weights)
        if freeze_backbone:
            for p in model.parameters(): p.requires_grad = False
        in_f = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_f, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes),
        )
    elif model_name == 'resnet50':
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        model   = models.resnet50(weights=weights)
        if freeze_backbone:
            for p in model.parameters(): p.requires_grad = False
        in_f = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_f, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes),
        )
    else:
        raise ValueError(f'Unknown model: {model_name}')

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Model  : {model_name}')
    print(f'Params : {trainable:,} trainable / {total:,} total ({100*trainable/total:.1f}%)')
    return model


def unfreeze_layers(model, model_name, num_layers=2):
    if model_name == 'resnet50':
        layer_groups = [model.layer4, model.layer3, model.layer2, model.layer1]
        for lg in layer_groups[:num_layers]:
            for p in lg.parameters(): p.requires_grad = True
    elif model_name == 'mobilenetv2':
        blocks = list(model.features.children())
        for block in blocks[-num_layers:]:
            for p in block.parameters(): p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Unfroze {num_layers} layer(s) → {trainable:,} / {total:,} params trainable ({100*trainable/total:.1f}%)')
    return model

print('Model builder ready.')

## Cell 4 — Training (Phase 1: Head + Phase 2: Fine-tune)
⏱️ **Estimated time: ~30–45 min on T4 GPU**

In [ ]:
# ── Training & Validation Functions ─────────────────────
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, desc='  Train', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += out.argmax(1).eq(labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, 100 * correct / total

@torch.no_grad()
def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, desc='  Val  ', leave=False):
        images, labels = images.to(device), labels.to(device)
        out  = model(images)
        loss = criterion(out, labels)
        total_loss += loss.item() * images.size(0)
        correct    += out.argmax(1).eq(labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, 100 * correct / total

print('Training functions ready.')

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 1: Train classification head only
# Backbone frozen — only new FC layers update
# ═══════════════════════════════════════════════
print('='*60)
print(f'PHASE 1 — Training head ({MODEL_NAME})')
print('='*60)

model     = build_model(MODEL_NAME, NUM_CLASSES, pretrained=True, freeze_backbone=True)
model     = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

history        = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[], 'phase':[]}
best_val_acc   = 0.0
patience_ctr   = 0
phase1_epochs  = min(NUM_EPOCHS // 2, 10)

for epoch in range(1, phase1_epochs + 1):
    print(f'\nEpoch {epoch}/{phase1_epochs}  [Phase 1]')
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc = val_epoch(model, val_loader, criterion)
    scheduler.step(vl_loss)

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)
    history['phase'].append(1)

    print(f'  Train → Loss: {tr_loss:.4f}  Acc: {tr_acc:.2f}%')
    print(f'  Val   → Loss: {vl_loss:.4f}  Acc: {vl_acc:.2f}%')

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), MODEL_DIR / f'{MODEL_NAME}_best.pth')
        print(f'  💾 Best saved  (val_acc = {vl_acc:.2f}%)')
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'  ⏹️  Early stopping at epoch {epoch}')
            break

print(f'\n✅ Phase 1 done. Best val acc: {best_val_acc:.2f}%')

In [ ]:
# ═══════════════════════════════════════════════
# PHASE 2: Fine-tune last 2 backbone blocks
# Much lower LR so we don't destroy ImageNet features
# ═══════════════════════════════════════════════
print('='*60)
print(f'PHASE 2 — Fine-tuning backbone ({MODEL_NAME})')
print('='*60)

# Reload best Phase 1 weights
model.load_state_dict(torch.load(MODEL_DIR / f'{MODEL_NAME}_best.pth', weights_only=True))
model = unfreeze_layers(model, MODEL_NAME, num_layers=2)

backbone_params = [p for n,p in model.named_parameters()
                   if p.requires_grad and 'fc' not in n and 'classifier' not in n]
head_params     = [p for n,p in model.named_parameters()
                   if p.requires_grad and ('fc' in n or 'classifier' in n)]

optimizer = optim.Adam([
    {'params': backbone_params, 'lr': FINETUNE_LR},
    {'params': head_params,     'lr': FINETUNE_LR * 10},
])
scheduler    = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)
patience_ctr = 0
phase2_epochs = NUM_EPOCHS - phase1_epochs

for epoch in range(1, phase2_epochs + 1):
    print(f'\nEpoch {epoch}/{phase2_epochs}  [Phase 2]')
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc = val_epoch(model, val_loader, criterion)
    scheduler.step(vl_loss)

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)
    history['phase'].append(2)

    print(f'  Train → Loss: {tr_loss:.4f}  Acc: {tr_acc:.2f}%')
    print(f'  Val   → Loss: {vl_loss:.4f}  Acc: {vl_acc:.2f}%')

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), MODEL_DIR / f'{MODEL_NAME}_best.pth')
        print(f'  💾 Best saved  (val_acc = {vl_acc:.2f}%)')
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'  ⏹️  Early stopping at epoch {epoch}')
            break

# Save training history
with open(MODEL_DIR / f'{MODEL_NAME}_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\n🏆 Training complete!')
print(f'   Best Val Accuracy : {best_val_acc:.2f}%')
print(f'   Model saved to    : {MODEL_DIR}/{MODEL_NAME}_best.pth')

## Cell 5 — Evaluation & Plots

In [ ]:
# ── Training Curves ──────────────────────────────────────
epochs = range(1, len(history['train_loss']) + 1)
phases = history['phase']
phase2_start = next((i+1 for i,p in enumerate(phases) if p==2), None)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, tr, vl, ylabel, title in [
    (ax1, history['train_loss'], history['val_loss'], 'Loss',         f'{MODEL_NAME} — Loss'),
    (ax2, history['train_acc'],  history['val_acc'],  'Accuracy (%)', f'{MODEL_NAME} — Accuracy'),
]:
    ax.plot(epochs, tr, 'b-o', markersize=3, label='Train')
    ax.plot(epochs, vl, 'r-o', markersize=3, label='Val')
    if phase2_start:
        ax.axvline(x=phase2_start, color='gray', linestyle='--', alpha=0.7, label='Fine-tune start')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'training_curves_{MODEL_NAME}.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved')

In [ ]:
# ── Test Set Evaluation ──────────────────────────────────
print('Running test set evaluation...')
model.load_state_dict(torch.load(MODEL_DIR / f'{MODEL_NAME}_best.pth', weights_only=True))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Testing'):
        images = images.to(device)
        preds  = model(images).argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
accuracy   = 100 * (all_preds == all_labels).mean()

class_names_display = [
    idx_to_class[i].replace('___', ' — ').replace('__', ' — ').replace('_', ' ')
    for i in range(NUM_CLASSES)
]

print(f'\n🎯 Test Accuracy: {accuracy:.2f}%\n')
report = classification_report(all_labels, all_preds, target_names=class_names_display)
print(report)

with open(OUTPUT_DIR / 'classification_report.txt', 'w') as f:
    f.write(f'Test Accuracy: {accuracy:.2f}%\n\n{report}')
print('✅ Classification report saved')

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

short_names = [
    idx_to_class[i].replace('___','\n').replace('__','\n').replace('_',' ')[:22]
    for i in range(NUM_CLASSES)
]

fig, ax = plt.subplots(figsize=(20, 16))
sns.heatmap(cm_norm, annot=False, cmap='Blues',
            xticklabels=short_names, yticklabels=short_names, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'{MODEL_NAME} — Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
plt.xticks(fontsize=6, rotation=90)
plt.yticks(fontsize=6, rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'confusion_matrix_{MODEL_NAME}.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved')

In [ ]:
# ── Grad-CAM Visualization ────────────────────────────────
import cv2
from PIL import Image

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.model.eval()
        self.grads = None
        self.acts  = None
        target_layer.register_forward_hook(lambda m,i,o: setattr(self, 'acts', o.detach()))
        target_layer.register_full_backward_hook(lambda m,gi,go: setattr(self, 'grads', go[0].detach()))

    def generate(self, inp, cls=None):
        out  = self.model(inp)
        cls  = cls or out.argmax(1).item()
        self.model.zero_grad()
        out[0, cls].backward()
        w    = self.grads.mean(dim=(2,3), keepdim=True)
        cam  = torch.relu((w * self.acts).sum(1)).squeeze().cpu().numpy()
        cam  = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, cls, out

target_layer = model.features[-1] if MODEL_NAME == 'mobilenetv2' else model.layer4[-1]
grad_cam     = GradCAM(model, target_layer)

# Sample one image per class for visualization
all_class_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])
sample_paths   = []
for d in all_class_dirs:
    imgs = list(d.glob('*.jpg')) + list(d.glob('*.JPG'))
    if imgs:
        sample_paths.append(np.random.choice(imgs))

np.random.shuffle(sample_paths)
n_show = min(8, len(sample_paths))

fig, axes = plt.subplots(n_show, 3, figsize=(12, n_show * 3))
for i, img_path in enumerate(sample_paths[:n_show]):
    img       = Image.open(img_path).convert('RGB')
    img_rsz   = img.resize((IMG_SIZE, IMG_SIZE))
    inp       = val_transforms(img).unsqueeze(0).to(device)
    cam, pred_idx, out = grad_cam.generate(inp)
    conf      = torch.softmax(out, 1)[0, pred_idx].item()
    cam_rsz   = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    heatmap   = plt.cm.jet(cam_rsz)[:, :, :3]
    overlay   = 0.5 * np.array(img_rsz) / 255.0 + 0.5 * heatmap

    true_name = img_path.parent.name
    pred_name = idx_to_class[pred_idx]
    _, true_c = get_plant_and_condition(true_name)
    _, pred_c = get_plant_and_condition(pred_name)

    axes[i][0].imshow(img_rsz);               axes[i][0].set_title(f'True: {true_c}', fontsize=8)
    axes[i][1].imshow(cam_rsz, cmap='jet');   axes[i][1].set_title('Grad-CAM',          fontsize=8)
    axes[i][2].imshow(overlay.clip(0,1));     axes[i][2].set_title(f'Pred: {pred_c} ({conf:.0%})', fontsize=8,
                                                                    color='green' if true_name==pred_name else 'red')
    for ax in axes[i]: ax.axis('off')

plt.suptitle(f'{MODEL_NAME} — Grad-CAM Visualizations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'gradcam_{MODEL_NAME}.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grad-CAM saved')

## Cell 6 — Download Model Files
Download these to your local `models/` folder — the Streamlit app needs them.

In [ ]:
# ── Download everything you need locally ─────────────────
from google.colab import files
import os

print('📥 Downloading trained model and class mapping...')
print('   Save both files into your local  models/  folder.\n')

files.download(str(MODEL_DIR / f'{MODEL_NAME}_best.pth'))
files.download(str(MODEL_DIR / 'class_mapping.json'))

print('\n📥 Downloading evaluation outputs...')
for fname in [
    f'training_curves_{MODEL_NAME}.png',
    f'confusion_matrix_{MODEL_NAME}.png',
    f'gradcam_{MODEL_NAME}.png',
    'classification_report.txt',
]:
    fpath = OUTPUT_DIR / fname
    if fpath.exists():
        files.download(str(fpath))
        print(f'   ✅ {fname}')

print('\n🎉 Done! Place the .pth and .json files in your local models/ folder,')
print('   then run: streamlit run streamlit_app/app.py')